VAE viewer notebook for JetBot
===

This notebook can visualize reconstructioned image by vae. This repository using JetBot camera.

In [ ]:
import sys
import PIL
import numpy as np
import cv2
import traitlets
import ipywidgets.widgets as widgets
from IPython.display import display
import torch
from torchvision.transforms import transforms
from jetbot import Camera, bgr8_to_jpeg
from learning_racer.vae import VAE

## Setting Parameter

|Name | Description| Default|
|:----|:-----------|:-------|
|IMAGE_CHANNELS | Image channel such as RGB | 3 Not change|
|VARIANTS_SIZE  | Variants size of VAE      | 32          |
|MODEL_PATH     | Trained VAE model file path | ../../vae.torch|

In [ ]:
IMAGE_CHANNELS = 3
VARIANTS_SIZE = 128
MODEL_PATH = '../../../vae_improved5.torch'

## Load trained VAE model.
Loading trained VAE model on GPU memory. 

In [ ]:
device = torch.device('cuda')
vae = VAE(image_channels=IMAGE_CHANNELS, z_dim=VARIANTS_SIZE)
vae.load_state_dict(torch.load(MODEL_PATH, map_location=torch.device(device)), strict=False)
vae.to(device).eval()

In [ ]:
from torchvision.models import vgg16, VGG16_Weights
import torch.nn.functional as F
import torchvision.transforms as transforms  # Note: not v2, to match viewer

# Load VGG for perceptual loss (features up to layer 16)
vgg = vgg16(weights=VGG16_Weights.DEFAULT).features[:16].eval().to(device)
for param in vgg.parameters():
    param.requires_grad = False

# Normalization for VGG (ImageNet stats)
vgg_normalization = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

## Create camera 


In [ ]:
camera = Camera.instance(width=320, height=240)

## Define preprocess and postprocess

In [ ]:
def preprocess(image):
    observe = PIL.Image.fromarray(image)
    observe = observe.resize((160,120))
    croped = observe.crop((0, 40, 160, 120))
    tensor = transforms.ToTensor()(croped)  # [0,1]
    
    # Optional: Test normalization like training (comment out if not needed)
    # tensor = transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])(tensor)  # To [-1,1]
    # Then in vae_process, un-normalize: image = image * 0.5 + 0.5
    
    return tensor
    

def rgb8_to_jpeg(image):
    return bytes(cv2.imencode('.jpg', image)[1])

## Visualize latent space function

In [ ]:
ABS_LATENT_MAX_VALUE = 3
PANEL_HEIGHT = 10
PANEL_WIDTH = 10

def sigmoid(x, gain=1, offset_x=0):
    return ((np.tanh(((x+offset_x)*gain)/2)+1)/2)

def color_bar_rgb(x):
    gain = 10
    offset_x= 0.2
    offset_green = 0.6
    x = (x * 2) - 1
    red = sigmoid(x, gain, -1*offset_x)
    blue = 1-sigmoid(x, gain, offset_x)
    green = sigmoid(x, gain, offset_green) + (1-sigmoid(x,gain,-1*offset_green))
    green = green - 1.0
    return [blue * 255,green * 255,red * 255]

def _get_color(value):
    t = (value + ABS_LATENT_MAX_VALUE) / (ABS_LATENT_MAX_VALUE * 2.0)
    color = color_bar_rgb(t)
    return color

def create_color_panel(latent_spaces):
    images = []
    for z in latent_spaces:
        p = np.zeros((PANEL_HEIGHT, PANEL_WIDTH, 3))
        color = _get_color(z)
        p += color[::-1]
        p = np.clip(p, 0, 255)
        images.append(p)
    panel = np.concatenate(images, axis=1)
    return panel

#Create GUI

In [ ]:
image = widgets.Image(format='jpeg', width=320, height=240)
resize = widgets.Image(format='jpeg', width=160, height=80)
result = widgets.Image(format='jpeg', width=160, height=80)
diff_image = widgets.Image(format='jpeg', width=160, height=80)  # New: Error map
sample_image = widgets.Image(format='jpeg', width=160, height=80)  # New: Random sample
camera_link = traitlets.dlink((camera,'value'), (image,'value'), transform=bgr8_to_jpeg)
color_bar = widgets.Image(format='jpeg', width=VARIANTS_SIZE*PANEL_WIDTH, height=10*PANEL_HEIGHT)

# New: Loss component displays
rec_loss_widget = widgets.FloatText(description='REC Loss:')
kld_loss_widget = widgets.FloatText(description='KLD:')
perc_loss_widget = widgets.FloatText(description='PERC Loss:')
total_loss_widget = widgets.FloatText(description='Total Loss:')

display(image)
display(widgets.HBox([resize, result, diff_image]))  # Add diff to row
display(color_bar)
display(widgets.HBox([rec_loss_widget, kld_loss_widget, perc_loss_widget, total_loss_widget]))
display(sample_image)  # Display random sample below

## Start main process

In [ ]:
def perceptual_loss(recon_x, x):
    x_norm = vgg_normalization(x)
    recon_x_norm = vgg_normalization(recon_x)
    feat_recon = vgg(recon_x_norm)
    feat_x = vgg(x_norm)
    return F.mse_loss(feat_recon, feat_x, reduction='mean')

def vae_process(change):
    image_np = change['new']
    image = preprocess(image_np)
    resize.value = rgb8_to_jpeg(np.transpose(np.uint8(image*255),[1,2,0]))
    
    # Optional un-normalize if you tested normalization in preprocess
    # image = image * 0.5 + 0.5
    # image = torch.clamp(image, 0, 1)
    
    image = torch.unsqueeze(image, dim=0).to(device)
    
    # Encode (adjust unpacking if your VAE.encode returns only mu, logvar)
    # If error here, change to: mu, logvar = vae.encode(image); z = vae.reparameterize(mu, logvar)
    z, mu, logvar = vae.encode(image)
    
    reconst = vae.decode(z)
    
    # Visualize reconstruction
    to_visualize = torch.squeeze(reconst).detach().cpu().numpy()
    to_visualize = np.transpose(np.uint8(to_visualize*255),[1,2,0])[:,:,::-1]
    result.value = rgb8_to_jpeg(to_visualize)
    
    # Latent space color bar
    latent_space = z.detach().cpu().numpy()[0]
    color_bar.value = rgb8_to_jpeg(create_color_panel(latent_space))
    
    # New: Print latent stats (check for collapse: logvar ~0, mu ~0, std(mu)~1)
    print("Mu: mean={:.4f}, std={:.4f}, min={:.4f}, max={:.4f}".format(
        mu.mean().item(), mu.std().item(), mu.min().item(), mu.max().item()))
    print("Logvar: mean={:.4f}, std={:.4f}, min={:.4f}, max={:.4f}".format(
        logvar.mean().item(), logvar.std().item(), logvar.min().item(), logvar.max().item()))
    print("Z: mean={:.4f}, std={:.4f}".format(z.mean().item(), z.std().item()))
    
    # New: Compute full loss components (use BCE to match training)
    REC = F.binary_cross_entropy(reconst, image, reduction='sum').item()  # Or MSE if preferred
    KLD = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp()).item()
    p_loss = perceptual_loss(reconst, image)
    PERC = p_loss.item() * reconst.numel()  # Scale like training
    total_loss = REC + 4.0 * KLD + 0.05 * PERC  # Use max beta from training
    
    # Update widgets
    rec_loss_widget.value = REC
    kld_loss_widget.value = KLD
    perc_loss_widget.value = PERC
    total_loss_widget.value = total_loss
    
    # New: Error map (abs diff, scaled for visibility)
    diff = torch.abs(reconst - image).mean(dim=1, keepdim=True)  # Grayscale avg over channels
    diff = diff / diff.max() * 255  # Normalize and scale
    diff_np = torch.squeeze(diff).detach().cpu().numpy()
    diff_np = cv2.cvtColor(np.uint8(diff_np), cv2.COLOR_GRAY2BGR)  # To 3-channel for JPEG
    diff_image.value = rgb8_to_jpeg(diff_np)
    
    # New: Generate random sample to test decoder (every 10 frames to save compute?)
    if np.random.rand() < 0.1:  # 10% chance per frame
        sample_z = torch.randn(1, VARIANTS_SIZE).to(device)  # Sample from N(0,1)
        sample_reconst = vae.decode(sample_z)
        sample_vis = torch.squeeze(sample_reconst).detach().cpu().numpy()
        sample_vis = np.transpose(np.uint8(sample_vis*255),[1,2,0])[:,:,::-1]
        sample_image.value = rgb8_to_jpeg(sample_vis)

vae_process({'new': camera.value})
camera.observe(vae_process, names='value')

## Cleanup process

In [ ]:
camera.unobserve(vae_process, names='value')
camera.stop()
camera_link.unlink()